# The Impact of Macroeconomic and Environmental Factors on Global Life Expectancy

## 1. Problem Formulation
This project investigates how economic prosperity (GDP per capita) and environmental degradation (PM2.5 air pollution) influence life expectancy across different nations. Understanding these relationships is crucial for global health policy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

who_df = pd.read_csv('data/raw/Life Expectancy Data.csv')

wb_gdp_df = pd.read_csv('data/raw/world_bank_gdp.csv', skiprows=4)
wb_pm25_df = pd.read_csv('data/raw/world_bank_pm25.csv', skiprows=4)

display(who_df.head(2))

,Country,Year,Status,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,...,Polio,Total expenditure,Diphtheria,HIV/AIDS,GDP,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,...,6.0,8.16,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,...,58.0,8.18,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0


In [2]:
# Strip leading and trailing spaces from all column names
who_df.columns = who_df.columns.str.strip()

# Drop any rows where the target variable ('Life expectancy') is missing
who_df = who_df.dropna(subset=['Life expectancy'])

print(f"WHO data shape after cleaning: {who_df.shape}")

WHO data shape after cleaning: (2928, 22)


In [3]:
# Identify the year columns (e.g., '1960', '2015')
gdp_years = [col for col in wb_gdp_df.columns if col.isnumeric()]
pm25_years = [col for col in wb_pm25_df.columns if col.isnumeric()]

# 1. Reshape and clean the GDP data
wb_gdp_clean = wb_gdp_df.melt(id_vars=['Country Name'], value_vars=gdp_years, var_name='Year', value_name='GDP_per_capita')
wb_gdp_clean = wb_gdp_clean.rename(columns={'Country Name': 'Country'})
wb_gdp_clean['Year'] = wb_gdp_clean['Year'].astype(int)

# 2. Reshape and clean the PM2.5 data
wb_pm25_clean = wb_pm25_df.melt(id_vars=['Country Name'], value_vars=pm25_years, var_name='Year', value_name='PM2.5_pollution')
wb_pm25_clean = wb_pm25_clean.rename(columns={'Country Name': 'Country'})
wb_pm25_clean['Year'] = wb_pm25_clean['Year'].astype(int)

# Display the cleaned GDP data to verify
display(wb_gdp_clean.head(2))

,Country,Year,GDP_per_capita
0,Aruba,1960,NaN
1,Africa Eastern and Southern,1960,186.089204


In [4]:
# First, merge the WHO data with the World Bank GDP data
merged_df = pd.merge(who_df, wb_gdp_clean, on=['Country', 'Year'], how='inner')

# Next, merge that result with the World Bank PM2.5 data
merged_df = pd.merge(merged_df, wb_pm25_clean, on=['Country', 'Year'], how='inner')

# Drop any rows that have missing economic or pollution data to keep our machine learning models clean
merged_df = merged_df.dropna(subset=['GDP_per_capita', 'PM2.5_pollution'])

print(f"Final merged dataset shape: {merged_df.shape}")
display(merged_df.head(2))

Final merged dataset shape: (2500, 24)


,Country,Year,Status,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,...,Diphtheria,HIV/AIDS,GDP,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling,GDP_per_capita,PM2.5_pollution
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,...,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1,565.569730,73.490818
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,...,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0,625.054942,77.143728
